In [ ]:
# ================================# 06-results-analysis# Phase 4: aggregation, CIs, learning curves, crossover detection# ================================import pandas as pdimport numpy as npimport matplotlib.pyplot as pltdf = pd.read_csv("/kaggle/input/notebooks/venkatkolluu/05-full-experiment-sweep/experiment_results.csv")print("Total rows:", len(df))print(df.head())

In [ ]:
# ================================# Document the collapse pattern explicitly -- don't hide it# ================================collapsed = df[df["accuracy"].round(4) == 0.3333]print(f"Runs at random-chance collapse: {len(collapsed)} / {len(df)} ({100*len(collapsed)/len(df):.1f}%)")print("Collapse breakdown by method:")print(collapsed.groupby("method").size())print("Collapse breakdown by (method, budget):")print(collapsed.groupby(["method", "budget"]).size())collapsed.to_csv("/kaggle/working/collapsed_runs.csv", index=False)

In [ ]:
# ================================# Aggregate: mean, std, 95% CI per (method, language, budget)# ================================N_SEEDS = 3summary = df.groupby(["method", "language", "budget"]).agg(    f1_mean=("macro_f1", "mean"),    f1_std=("macro_f1", "std"),    acc_mean=("accuracy", "mean"),    acc_std=("accuracy", "std"),    n=("seed", "count"),).reset_index()summary["f1_ci95"] = 1.96 * summary["f1_std"].fillna(0) / np.sqrt(summary["n"])summary["acc_ci95"] = 1.96 * summary["acc_std"].fillna(0) / np.sqrt(summary["n"])summary = summary.sort_values(["language", "method", "budget"])summary.to_csv("/kaggle/working/summary_with_ci.csv", index=False)display(summary)

In [ ]:
# ================================# Learning curve plots -- one per language, F1 vs budget, CI bands# ================================BUDGETS = [50, 100, 500, 1000, 2000, 20000]METHODS = ["lora", "dora", "ia3"]COLORS = {"lora": "#1f77b4", "dora": "#ff7f0e", "ia3": "#2ca02c"}fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)for ax, language, title in zip(axes, ["hi", "te"], ["Hindi", "Telugu"]):    for method in METHODS:        sub = summary[(summary.method == method) & (summary.language == language)].sort_values("budget")        ax.plot(sub["budget"], sub["f1_mean"], marker="o", label=method.upper(), color=COLORS[method])        ax.fill_between(sub["budget"], sub["f1_mean"] - sub["f1_ci95"], sub["f1_mean"] + sub["f1_ci95"],                         alpha=0.15, color=COLORS[method])    ax.set_xscale("log")    ax.set_xlabel("Training budget (samples, log scale)")    ax.set_title(f"{title} -- Macro F1 vs. Budget")    ax.axhline(0.333, color="gray", linestyle="--", linewidth=1, label="Random chance")    ax.grid(alpha=0.3)axes[0].set_ylabel("Macro F1")axes[0].legend(loc="lower right", fontsize=9)plt.tight_layout()plt.savefig("/kaggle/working/learning_curves.png", dpi=150)plt.show()

In [ ]:
# ================================# Crossover-point detection# ================================def detect_crossovers(summary, language):    lang_df = summary[summary.language == language].copy()    budgets_sorted = sorted(lang_df.budget.unique())    crossovers = []    for i in range(len(budgets_sorted) - 1):        b1, b2 = budgets_sorted[i], budgets_sorted[i + 1]        row1 = lang_df[lang_df.budget == b1].set_index("method")        row2 = lang_df[lang_df.budget == b2].set_index("method")        rank1 = row1["f1_mean"].sort_values(ascending=False).index.tolist()        rank2 = row2["f1_mean"].sort_values(ascending=False).index.tolist()        if rank1 != rank2:            crossovers.append({"language": language, "budget_before": b1, "budget_after": b2,                                "ranking_before": rank1, "ranking_after": rank2})    return crossovershi_crossovers = detect_crossovers(summary, "hi")te_crossovers = detect_crossovers(summary, "te")print("Hindi crossovers:")for c in hi_crossovers:    print(f"  budget={c[\'budget_before\']} -> {c[\'budget_after\']}: {c[\'ranking_before\']} -> {c[\'ranking_after\']}")print("Telugu crossovers:")for c in te_crossovers:    print(f"  budget={c[\'budget_before\']} -> {c[\'budget_after\']}: {c[\'ranking_before\']} -> {c[\'ranking_after\']}")

In [ ]:
# ================================# Ranking table -- best method per (language, budget)# ================================ranking_table = summary.pivot_table(index=["language", "budget"], columns="method", values="f1_mean")ranking_table["best_method"] = ranking_table[METHODS].idxmax(axis=1)display(ranking_table)ranking_table.to_csv("/kaggle/working/ranking_table.csv")

In [ ]:
# ================================# Compute-efficiency comparison# ================================compute_summary = df.groupby("method").agg(    trainable_params=("trainable_params", "first"),    avg_peak_memory_gb=("peak_gpu_memory_gb", "mean"),    total_training_time_sec=("training_time_sec", "sum"),).reset_index()compute_summary["total_training_time_min"] = compute_summary["total_training_time_sec"] / 60display(compute_summary)compute_summary.to_csv("/kaggle/working/compute_efficiency.csv", index=False)

In [ ]:
# ================================# Does the ranking hold across languages? -- the core research question# ================================comparison = ranking_table.reset_index().pivot(index="budget", columns="language", values="best_method")display(comparison)agreement = (comparison["hi"] == comparison["te"]).mean()print(f"Ranking agreement across languages: {agreement*100:.1f}% of budgets")